# Feature 015 — rolling FCR backtest walkthrough

This source-only notebook demonstrates how Open Energy Lab composes **Feature 013 bid optimization**, **Feature 012 physical replay and settlement**, and **Feature 015 rolling orchestration** across several delivery dates. It focuses on the five pinned non-intraday profiles, midnight state carry-over, the 08:00 overlapping horizon, causal information boundaries, record lifecycle, economics, and bounded Julia/Gurobi evidence.

The notebook answers a deliberately bounded question: can the native workflow reproduce the declared multi-day behavior while keeping optimization evidence, realized evaluation, and source compatibility distinct? It does **not** claim Table A1 parity, market acceptance evidence, full-history completion, or proven optimality for time-limited source incumbents.

## Setup, default workload, and evidence limits

From the repository root, install the optional notebook tools with `uv sync --group notebook`, then start the source notebook with `uv run jupyter lab examples/fcr_rolling_backtest/fcr_rolling_backtest_walkthrough.ipynb`. Select the project environment (`.venv/bin/python`) as the kernel. The committed notebook has stable cell IDs, empty outputs, and no execution counts.

Default execution is deterministic and offline. It builds all five profile manifests, solves the lightweight four-day Script 13 authored study, replays it through Feature 012, and inspects the committed normalized source fixture. Solving all five authored profiles is disabled because it is a research-scale operation: several mixed-binary cases can consume their complete per-day time limits. The native reference benchmark, external activation verification, and figure export are also explicit opt-ins. No cell downloads data, searches for external files, invokes Julia, or reads solver credentials.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
DEFAULT_PROFILE_ID = "script-13-arbitrage-only-midnight-v1"
RUN_ALL_AUTHORED_PROFILES = False
RUN_NATIVE_REFERENCE_BENCHMARK = False
RUN_EXTERNAL_ACTIVATION_REPLAY = False
EXPORT_FIGURES = False
AUTHORED_TIME_LIMIT_SECONDS = 10.0
REFERENCE_BENCHMARK_TIME_LIMIT_SECONDS = 60.0
REFERENCE_FIXTURE_PATH = Path("tests/fixtures/fcr_rolling_backtest/robust_storage_bidding_experiments_01_05_2023-06-17_20_v1.json")
EXTERNAL_ACTIVATION_CSV = Path("artifacts/reference/robust_storage_bidding/regulation_signal.csv")
FIGURE_EXPORT_DIR = Path("tmp/fcr-rolling-backtest-notebook/figures")

In [ ]:
import hashlib
import json

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import pandas as pd
from IPython.display import display

from open_energy_lab.control import (
    RollingFCRStudyResult,
    derive_rolling_fcr_state_carry,
)
from oel_validation.fcr_rolling_reference import (
    load_rolling_fcr_reference_fixture,
)
from examples.fcr_rolling_backtest.run_authored_study import run_authored_profiles
from examples.fcr_rolling_backtest.study_inputs import (
    BOUNDED_DATES,
    PROFILE_IDS,
    build_authored_activation_traces,
    build_manifest,
    build_solve_policy,
    fingerprint,
)

plt.rcParams.update({"figure.figsize": (10, 4.8), "axes.grid": True, "grid.alpha": 0.25})
generated_figures = {}

def stable_fingerprint(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def script_label(profile_id):
    return "Script " + profile_id.removeprefix("script-").split("-", 1)[0]

## 1. Ownership and the robust-budget lineage

Feature 013 owns the ex-ante bid problem and returns hourly charge-positive energy decisions plus symmetric FCR capacity. Feature 012 is the only owner of battery transitions, tracking, settlement, and realized profit. Feature 015 only orders dates, applies cutoffs and accepted fixings, carries state, records immutable revisions, and aggregates evidence.

The recovery-window set and horizon-wide set are distinct. For horizon $T$, recovery window $\Gamma'$, and recovery activation budget $\gamma'$, the non-intraday budget is

$$n=\left\lfloor\frac{T}{\Gamma'}\right\rfloor,\qquad \gamma=n\gamma' + \min(T-n\Gamma',\gamma').$$

With $T=24\,\mathrm{h}$, $\Gamma'=2.25\,\mathrm{h}$, and $\gamma'=0.25\,\mathrm{h}$, this gives $\gamma=2.75\,\mathrm{h}$. Activation is evaluation-only data and is never an input to bid selection.

In [ ]:
authored_solve_policy = build_solve_policy(time_limit_seconds=AUTHORED_TIME_LIMIT_SECONDS)
profile_manifests = {
    profile_id: build_manifest(profile_id, solve_policy=authored_solve_policy)
    for profile_id in PROFILE_IDS
}
reference_manifest = profile_manifests[DEFAULT_PROFILE_ID]
uncertainty = reference_manifest.uncertainty_profile
budget_table = pd.DataFrame([
    {"quantity": "recovery activation budget", "symbol": "gamma prime", "value": uncertainty.recovery_window.recovery_activation_budget_hours, "unit": "h"},
    {"quantity": "recovery window", "symbol": "Gamma prime", "value": uncertainty.recovery_window.recovery_window_hours, "unit": "h"},
    {"quantity": "derived horizon budget", "symbol": "gamma", "value": uncertainty.horizon_wide.horizon_activation_budget_hours, "unit": "h"},
])
assert budget_table.iloc[-1]["value"] == 2.75
display(budget_table)

In [ ]:
profile_table = pd.DataFrame([
    {
        "profile": script_label(profile_id),
        "formulation": manifest.profile.formulation_kind,
        "state mode": manifest.profile.state_mode,
        "bid hour (local)": manifest.profile.bidding_time_local_hour,
        "FCR rule": "fixed exactly to zero" if manifest.profile.formulation_kind == "arbitrage_only" else "optimized symmetric capacity",
        "plan classification": manifest.profile.plan_classification,
        "carry policies": ", ".join(manifest.profile.allowed_state_carry_policies),
        "failure policy": manifest.failure_policy,
        "Table A1 parity": manifest.profile.table_a1_parity_claim,
    }
    for profile_id, manifest in profile_manifests.items()
])
display(profile_table)

## 2. Offline authored study

The default profile is Script 13: four ordinary days of day-ahead arbitrage with every FCR decision constrained exactly to zero and realized midnight state carried into the successor. This is a real Pyomo/HiGHS solve followed by Feature 012 replay; it is not Julia evidence. Set `RUN_ALL_AUTHORED_PROFILES = True` only when a longer mixed-binary research run is intended. Relaxation results remain diagnostic and are never labelled robustly deployable.

In [ ]:
selected_authored_profiles = PROFILE_IDS if RUN_ALL_AUTHORED_PROFILES else (DEFAULT_PROFILE_ID,)
authored_bundle = run_authored_profiles(
    profile_ids=selected_authored_profiles,
    state_carry_policy="source_three_decimal",
    time_limit_seconds=AUTHORED_TIME_LIMIT_SECONDS,
)
authored_results = {
    item["profile_id"]: RollingFCRStudyResult.from_dict(item["result"])
    for item in authored_bundle["profiles"]
}
authored_result = authored_results[DEFAULT_PROFILE_ID]
assert authored_result.completeness_status == "complete"
assert len(authored_result.records) == 4
authored_summary_table = pd.DataFrame([
    {"profile": script_label(profile_id), "completeness": result.completeness_status, "dates": len(result.records), "semantic fingerprint": result.semantic_fingerprint}
    for profile_id, result in authored_results.items()
])
display(authored_summary_table)

In [ ]:
authored_daily_rows = []
for envelope in authored_result.records:
    revision = envelope.latest_revision
    authored_daily_rows.append({
        "delivery date": envelope.classification.delivery_date.isoformat(),
        "lifecycle": revision.lifecycle_status,
        "objective (EUR)": revision.objective_value_eur,
        "best upper bound (EUR)": revision.best_upper_bound_eur,
        "initial stored energy (kWh)": revision.initial_stored_energy_kwh,
        "realized terminal energy (kWh)": revision.realized_terminal_stored_energy_kwh,
        "carried energy (kWh)": revision.carried_stored_energy_kwh,
        "realized settlement (EUR)": revision.realized_metrics.get("settlement.total_profit_eur"),
        "realized profit (EUR)": revision.realized_metrics.get("total_profit_eur"),
        "plan class": revision.plan_classification,
    })
authored_daily_table = pd.DataFrame(authored_daily_rows)
state_link_table = pd.DataFrame([link.to_dict() for link in authored_result.state_links])
display(authored_daily_table)
display(state_link_table)

In [ ]:
schedule_revision = authored_result.records[1].latest_revision
energy_schedule_kw = schedule_revision.energy_decisions_charge_positive_kw
fcr_schedule_kw = schedule_revision.fcr_capacity_kw
assert energy_schedule_kw is not None and fcr_schedule_kw is not None
assert all(value == 0.0 for value in fcr_schedule_kw)
fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, constrained_layout=True)
axes[0].bar(range(24), energy_schedule_kw, width=0.85, color="#2a6fbb")
axes[0].axhline(0.0, color="black", linewidth=0.8)
axes[0].set_ylabel("Energy baseline (kW)")
axes[0].set_title("Authored Script 13 schedule — charge-positive convention")
axes[1].bar(range(0, 24, 4), fcr_schedule_kw, width=3.8, align="edge", color="#ef8a17")
axes[1].set_ylabel("Symmetric FCR capacity (kW)")
axes[1].set_xlabel("Delivery hour (Europe/Paris)")
axes[1].set_xticks(range(0, 25, 4))
axes[1].set_title("FCR is constrained exactly to zero, not induced by zero prices")
generated_figures["authored_energy_fcr_schedules"] = fig
plt.show()

## 3. Midnight carry-over and the state ledger

The optimizer's modeled terminal state is not permitted to initialize the next day. Feature 015 first accepts and physically replays the plan through Feature 012, then carries the **realized** terminal state. For delivery day $d$,

$$E^{\mathrm{init}}_{d+1}=Q_p\!\left(E^{\mathrm{realized}}_{d,24\mathrm{h}}\right),$$

where $Q_p$ is the manifest's state-carry policy. Every value below is stored energy in **kWh**; there is no dimensionless deviation axis.

In [ ]:
state_ledger_table = authored_daily_table[[
    "delivery date",
    "initial stored energy (kWh)",
    "realized terminal energy (kWh)",
    "carried energy (kWh)",
]].copy()
state_ledger_table["successor reconciliation residual (kWh)"] = [
    None,
    *[
        state_ledger_table.iloc[index]["initial stored energy (kWh)"]
        - state_ledger_table.iloc[index - 1]["carried energy (kWh)"]
        for index in range(1, len(state_ledger_table))
    ],
]
assert state_ledger_table["successor reconciliation residual (kWh)"].dropna().abs().max() <= 1e-9
display(state_ledger_table)

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
dates = state_ledger_table["delivery date"]
ax.plot(dates, state_ledger_table["initial stored energy (kWh)"], marker="o", label="initial")
ax.plot(dates, state_ledger_table["realized terminal energy (kWh)"], marker="s", label="realized terminal")
ax.plot(dates, state_ledger_table["carried energy (kWh)"], marker="^", label="carried to successor")
ax.set_ylabel("Stored energy (kWh)")
ax.set_xlabel("Delivery date")
ax.set_title("Feature 012 realized state and Feature 015 midnight handoff")
ax.legend(ncol=3)
generated_figures["midnight_state_ledger"] = fig
plt.show()

### Source-compatible rounding versus native full precision

The source-compatible policy uses decimal half-even quantization, $Q_3(E)=10^{-3}\operatorname{round}_{\mathrm{half-even}}(10^3E)$, and records the closed precision interval

$$I_3(Q_3(E))=[Q_3(E)-0.0005,\;Q_3(E)+0.0005]\;\mathrm{kWh}.$$

The native policy is the identity $Q_{\mathrm{native}}(E)=E$. The policies have different fingerprints and make different claims about later decisions. Only Script 15 permits native full precision; the 08:00 profile deliberately remains source-compatible.

In [ ]:
policy_checkpoint_kwh = 9.87654
carry_evidence = {
    policy: derive_rolling_fcr_state_carry(policy_checkpoint_kwh, state_carry_policy=policy)
    for policy in ("source_three_decimal", "native_full_precision")
}
carry_policy_table = pd.DataFrame([
    {
        "policy": policy,
        "replay checkpoint (kWh)": evidence.replay_checkpoint_kwh,
        "carried state (kWh)": evidence.carried_stored_energy_kwh,
        "rounding difference (kWh)": evidence.carried_stored_energy_kwh - evidence.replay_checkpoint_kwh,
        "closed lower bound (kWh)": evidence.source_interval_kwh[0],
        "closed upper bound (kWh)": evidence.source_interval_kwh[1],
        "evidence fingerprint": evidence.evidence_fingerprint,
    }
    for policy, evidence in carry_evidence.items()
])
assert carry_policy_table["evidence fingerprint"].nunique() == 2
display(carry_policy_table)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
for index, row in carry_policy_table.iterrows():
    value = row["carried state (kWh)"]
    lower = row["closed lower bound (kWh)"]
    upper = row["closed upper bound (kWh)"]
    ax.errorbar(index, value, yerr=[[value - lower], [upper - value]], fmt="o", capsize=7, markersize=8)
ax.axhline(policy_checkpoint_kwh, color="black", linestyle="--", label="Feature 012 replay checkpoint")
ax.set_xticks(range(len(carry_policy_table)), carry_policy_table["policy"])
ax.set_ylabel("Stored energy (kWh)")
ax.set_title("Carry value and explicit source-precision interval")
ax.legend()
generated_figures["state_carry_precision"] = fig
plt.show()

## 4. The 08:00 overlapping horizon

Script 14 solves a 40-hour frame at 08:00 local time: the accepted predecessor tail covers the next 16 hours and is fixed, while the new delivery day contributes 24 hourly energy decisions. Four predecessor FCR blocks plus six new blocks form ten four-hour blocks. If $x^{\mathrm{accepted}}$ is the accepted tail and $y$ is the new decision vector, the rolling problem enforces

$$x_{0:16}=x^{\mathrm{accepted}}_{0:16},\qquad r_{0:4}=r^{\mathrm{accepted}}_{0:4},$$

and optimizes only the remaining variables. The source JSON rounds displayed decisions to three decimals, so exact hidden full-precision fixings cannot be reconstructed from it. The figure below is source evidence, not a native parity claim.

In [ ]:
reference_fixture = load_rolling_fcr_reference_fixture(REPO_ROOT / REFERENCE_FIXTURE_PATH)
assert len(reference_fixture.profile_crosswalk) == 5
assert len(reference_fixture.solve_records) == 20
assert len(reference_fixture.delivery_evidence) == 20
reference_fixture_summary = pd.DataFrame([
    {"field": "fixture id", "value": reference_fixture.fixture_id},
    {"field": "reference revision", "value": reference_fixture.reference_provenance.reference_revision},
    {"field": "profiles", "value": len(reference_fixture.profile_crosswalk)},
    {"field": "solve records", "value": len(reference_fixture.solve_records)},
    {"field": "activation samples redistributed", "value": reference_fixture.redistribution_status.fixture_contains_activation_samples},
    {"field": "fixture fingerprint", "value": reference_fixture.fixture_fingerprint},
])
display(reference_fixture_summary)

In [ ]:
reference_crosswalk_table = pd.DataFrame([item.to_dict() for item in reference_fixture.profile_crosswalk])
reference_artifact_table = pd.DataFrame([item.to_dict() for item in reference_fixture.raw_input_manifest])
display(reference_crosswalk_table[["profile_id", "script", "result_directory", "paper_experiment", "workbook_column", "mapping_status", "table_a1_claim"]])
display(reference_artifact_table[["artifact_id", "artifact_kind", "sha256", "redistributed"]])

In [ ]:
overlap_profile_id = "script-14-restriction-rolling-0800-v1"
overlap_date = BOUNDED_DATES[2]
overlap_record = next(item for item in reference_fixture.solve_records if item.profile_id == overlap_profile_id and item.delivery_date == overlap_date)
accepted_overlap = overlap_record.accepted_overlap
assert accepted_overlap is not None
fixed_energy_kw = tuple(-value for value in accepted_overlap.hourly_energy_source_positive_discharge_kw)
new_energy_kw = tuple(-value for value in overlap_record.hourly_energy_source_positive_discharge_kw)
fixed_fcr_kw = tuple(accepted_overlap.fcr_capacity_kw)
new_fcr_kw = tuple(overlap_record.fcr_capacity_kw)
overlap_energy_table = pd.DataFrame([
    {"hour from 08:00 bid cutoff": hour, "segment": "fixed accepted predecessor" if hour < 16 else "new delivery decision", "energy baseline (kW, charge-positive)": value}
    for hour, value in enumerate((*fixed_energy_kw, *new_energy_kw))
])
overlap_fcr_table = pd.DataFrame([
    {"block start hour from cutoff": 4 * index, "segment": "fixed accepted predecessor" if index < 4 else "new delivery decision", "symmetric FCR capacity (kW)": value}
    for index, value in enumerate((*fixed_fcr_kw, *new_fcr_kw))
])
display(overlap_energy_table)
display(overlap_fcr_table)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
segment_colors = ["#888888" if hour < 16 else "#2a6fbb" for hour in range(40)]
axes[0].bar(range(40), overlap_energy_table["energy baseline (kW, charge-positive)"], color=segment_colors, width=0.9)
axes[0].axhline(0.0, color="black", linewidth=0.8)
axes[0].set_ylabel("Energy baseline (kW)")
axes[0].set_title("08:00 rolling horizon: accepted tail is fixed; new delivery is optimized")
fcr_colors = ["#888888" if index < 4 else "#ef8a17" for index in range(10)]
axes[1].bar(range(0, 40, 4), overlap_fcr_table["symmetric FCR capacity (kW)"], width=3.8, align="edge", color=fcr_colors)
axes[1].set_ylabel("Symmetric FCR capacity (kW)")
axes[1].set_xlabel("Hours from 08:00 bid cutoff")
for ax in axes:
    ax.axvline(16, color="#b2182b", linestyle="--", label="new delivery starts")
axes[0].legend(handles=[Patch(color="#888888", label="fixed accepted predecessor"), Patch(color="#2a6fbb", label="new hourly decisions"), Patch(color="#ef8a17", label="new FCR blocks"), Patch(color="#b2182b", label="delivery midnight")], ncol=2)
generated_figures["rolling_0800_overlap"] = fig
plt.show()

## 5. Causal information boundary

Let $t_c$ be the information cutoff and let $a$ and $a'$ be activation traces that agree on the half-open observed interval $[t_0,t_c)$ but differ afterward. The bidder-visible input must satisfy

$$a|_{[t_0,t_c)}=a'|_{[t_0,t_c)}\;\Longrightarrow\;I_{t_c}(a)=I_{t_c}(a')\;\Longrightarrow\;B(I_{t_c}(a))=B(I_{t_c}(a')).$$

The hand calculation below changes every post-cutoff sample while retaining the first eight hours. It checks the exact prefix identity that the production 08:00 orchestration contract uses. It does not claim that the source fixture contains the undistributed activation samples.

In [ ]:
causal_date = BOUNDED_DATES[1]
authored_trace = build_authored_activation_traces((causal_date,))[causal_date]
activation_a = tuple(sample.activation for sample in authored_trace.samples)
cutoff_sample_count = 8 * 4
activation_b = activation_a[:cutoff_sample_count] + tuple(0.75 if value == 0.0 else -value for value in activation_a[cutoff_sample_count:])
observed_a_fingerprint = fingerprint(activation_a[:cutoff_sample_count])
observed_b_fingerprint = fingerprint(activation_b[:cutoff_sample_count])
full_a_fingerprint = fingerprint(activation_a)
full_b_fingerprint = fingerprint(activation_b)
assert observed_a_fingerprint == observed_b_fingerprint
assert full_a_fingerprint != full_b_fingerprint
causality_table = pd.DataFrame([
    {"evidence": "observed [00:00, 08:00) prefix", "trace A fingerprint": observed_a_fingerprint, "trace B fingerprint": observed_b_fingerprint, "unchanged": observed_a_fingerprint == observed_b_fingerprint},
    {"evidence": "complete 24-hour activation", "trace A fingerprint": full_a_fingerprint, "trace B fingerprint": full_b_fingerprint, "unchanged": full_a_fingerprint == full_b_fingerprint},
])
display(causality_table)

## 6. Ex-ante optimization versus ex-post economics

The bid objective and same-solve upper bound are optimizer evidence available before delivery. Settlement and profit are Feature 012 results that require the accepted plan and realized activation:

$$z^{\mathrm{bid}}_d=R^{\mathrm{DA}}_d+R^{\mathrm{FCRcap}}_d-P^{\mathrm{terminal}}_d,\qquad \Pi^{\mathrm{realized}}_d=\operatorname{Settlement}_{012}(x_d,a_d).$$

These quantities must remain separately named even when they happen to be numerically close in the authored zero-FCR example. Both plot panels use EUR, but one is ex ante and the other ex post.

In [ ]:
economics_table = authored_daily_table[[
    "delivery date",
    "objective (EUR)",
    "best upper bound (EUR)",
    "realized settlement (EUR)",
    "realized profit (EUR)",
]].copy()
economics_table["objective-to-bound gap (EUR)"] = economics_table["best upper bound (EUR)"] - economics_table["objective (EUR)"]
aggregate_economics_table = pd.DataFrame([item.to_dict() for item in authored_result.aggregate_metrics if item.metric_id in {"optimization_objective_eur", "best_upper_bound_eur", "realized_settlement_eur", "realized_profit_eur"}])
display(economics_table)
display(aggregate_economics_table[["metric_id", "value", "unit", "coverage_status", "contributing_dates"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
x = list(range(len(economics_table)))
axes[0].bar([value - 0.18 for value in x], economics_table["objective (EUR)"], width=0.36, label="incumbent objective")
axes[0].bar([value + 0.18 for value in x], economics_table["best upper bound (EUR)"], width=0.36, label="same-solve upper bound")
axes[0].set_title("Ex-ante optimization evidence")
axes[0].set_ylabel("Value (EUR)")
axes[0].legend()
axes[1].bar([value - 0.18 for value in x], economics_table["realized settlement (EUR)"], width=0.36, label="settlement")
axes[1].bar([value + 0.18 for value in x], economics_table["realized profit (EUR)"], width=0.36, label="realized profit")
axes[1].set_title("Ex-post Feature 012 evidence")
axes[1].set_ylabel("Value (EUR)")
axes[1].legend()
for ax in axes:
    ax.set_xticks(x, economics_table["delivery date"], rotation=25)
    ax.set_xlabel("Delivery date")
generated_figures["optimization_and_realized_economics"] = fig
plt.show()

## 7. Original, provisional, finalized, and edge records

One immutable envelope remains present for every requested date. In the 08:00 profile, the first date initializes the accepted overlap, an ordinary date becomes final only when its successor closes it, and the last delivery remains unfinalized until a later solve exists. Warm-up and unfinalized dates stay visible but are excluded from complete parity aggregates with explicit reasons.

In [ ]:
lifecycle_table = pd.DataFrame([
    {
        "delivery date": item.delivery_date.isoformat(),
        "lifecycle": item.source_lifecycle_status,
        "predecessor": None if item.predecessor_delivery_date is None else item.predecessor_delivery_date.isoformat(),
        "original available": item.original_evidence_available,
        "finalized available": item.finalized_evidence_available,
        "comparison complete": item.source_lifecycle_status == "finalized",
    }
    for item in reference_fixture.delivery_evidence
    if item.profile_id == overlap_profile_id
])
display(lifecycle_table)

In [ ]:
lifecycle_order = ["initialization_only", "finalized", "unfinalized"]
lifecycle_colors = {"initialization_only": "#8073ac", "finalized": "#1b9e77", "unfinalized": "#d95f02"}
fig, ax = plt.subplots(figsize=(10, 4.6), constrained_layout=True)
for index, row in lifecycle_table.iterrows():
    y = lifecycle_order.index(row["lifecycle"])
    ax.scatter(index, y, s=180, color=lifecycle_colors[row["lifecycle"]], edgecolor="black")
    ax.text(index, y + 0.16, row["lifecycle"], ha="center", fontsize=9)
ax.set_xticks(range(len(lifecycle_table)), lifecycle_table["delivery date"])
ax.set_yticks(range(len(lifecycle_order)), lifecycle_order)
ax.set_xlabel("Delivery date")
ax.set_ylabel("Record lifecycle category")
ax.set_title("Bounded source 08:00 lifecycle — edge dates remain visible")
generated_figures["rolling_0800_lifecycle"] = fig
plt.show()

## 8. Bounded Julia/Gurobi source evidence

The normalized fixture covers five profiles and four dates at reference revision `7e44738bb30e5c8204efe40103b9be4c2203685c`. It stores displayed three-decimal schedules, objective and bound evidence, log-derived termination, provenance, and lifecycle—but not raw logs, the workbook, the full JSON collection, or activation samples.

A displayed value $v$ at three decimals denotes the closed source-precision envelope $[v-0.0005,v+0.0005]$ in that field's unit. This is not a feasibility tolerance. `bilinear_violations=0` is not independent feasibility proof. `TIME_LIMIT` means feasible-incumbent-plus-bound evidence, not proven optimality or an alternative optimum.

In [ ]:
source_solve_table = pd.DataFrame([
    {
        "profile": script_label(item.profile_id),
        "delivery date": item.delivery_date.isoformat(),
        "termination": item.solver_evidence.termination_condition,
        "incumbent available": item.solver_evidence.incumbent_available,
        "objective (EUR)": item.objective_eur,
        "best upper bound (EUR)": item.best_upper_bound_eur,
        "absolute gap (EUR)": item.absolute_gap_eur,
        "modeled feasibility claim": item.modeled_feasibility_claim,
    }
    for item in reference_fixture.solve_records
])
source_precision_table = pd.DataFrame([
    {"displayed value": value, "closed lower bound": value - 0.0005, "closed upper bound": value + 0.0005, "unit": unit}
    for value, unit in ((6.738, "EUR"), (11.736, "kW"), (57.988, "kWh"))
])
display(source_solve_table)
display(source_precision_table)

In [ ]:
termination_categories = ["OPTIMAL", "TIME_LIMIT"]
termination_matrix = []
for profile_id in PROFILE_IDS:
    row = []
    for delivery_date in BOUNDED_DATES:
        record = next(item for item in reference_fixture.solve_records if item.profile_id == profile_id and item.delivery_date == delivery_date)
        row.append(termination_categories.index(record.solver_evidence.termination_condition))
    termination_matrix.append(row)
fig, ax = plt.subplots(figsize=(10, 5.2), constrained_layout=True)
ax.imshow(termination_matrix, cmap=ListedColormap(["#1b9e77", "#d95f02"]), vmin=-0.5, vmax=1.5, aspect="auto")
for y, profile_id in enumerate(PROFILE_IDS):
    for x, delivery_date in enumerate(BOUNDED_DATES):
        ax.text(x, y, termination_categories[termination_matrix[y][x]], ha="center", va="center", color="white", fontweight="bold", fontsize=8)
ax.set_xticks(range(len(BOUNDED_DATES)), [item.isoformat() for item in BOUNDED_DATES])
ax.set_yticks(range(len(PROFILE_IDS)), [script_label(item) for item in PROFILE_IDS])
ax.set_xlabel("Delivery date")
ax.set_ylabel("Source executable profile")
ax.set_title("Julia/Gurobi termination evidence (categorical, not a numeric score)")
ax.legend(handles=[Patch(color="#1b9e77", label="OPTIMAL source termination"), Patch(color="#d95f02", label="TIME_LIMIT: incumbent plus bound")], loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
generated_figures["source_termination_evidence"] = fig
plt.show()

## 9. Failure propagation and resume identity

Independent reset profiles may continue after a failed day. Coupled midnight and 08:00 profiles cannot invent a missing physical state: dependent successors become `blocked_by_predecessor` unless an explicit fingerprinted restart state begins a new continuity segment. Resume requires exact agreement of the manifest, semantic profile, date order, solver and numeric policies, source and activation identities, completed-day fingerprints, and carry state. Runtime duration and caller-local output paths are intentionally non-semantic.

In [ ]:
failure_resume_table = pd.DataFrame([
    {
        "profile": script_label(profile_id),
        "state mode": manifest.profile.state_mode,
        "failure policy": manifest.failure_policy,
        "successor after missing state": "continue independently" if manifest.profile.is_independent else "blocked_by_predecessor",
        "resume policy": manifest.resume_policy,
    }
    for profile_id, manifest in profile_manifests.items()
])
failure_transition_table = pd.DataFrame([
    {"profile coupling": "independent reset", "predecessor outcome": "failed", "successor outcome": "executes independently", "missing state invented": False},
    {"profile coupling": "midnight or 08:00 carry", "predecessor outcome": "failed or activation unavailable", "successor outcome": "blocked_by_predecessor", "missing state invented": False},
])
partial_bundle = run_authored_profiles(
    profile_ids=(DEFAULT_PROFILE_ID,),
    state_carry_policy="source_three_decimal",
    time_limit_seconds=AUTHORED_TIME_LIMIT_SECONDS,
    stop_after=2,
)
partial_result = RollingFCRStudyResult.from_dict(partial_bundle["profiles"][0]["result"])
resumed_bundle = run_authored_profiles(
    profile_ids=(DEFAULT_PROFILE_ID,),
    state_carry_policy="source_three_decimal",
    time_limit_seconds=AUTHORED_TIME_LIMIT_SECONDS,
    resume_results={DEFAULT_PROFILE_ID: partial_result},
)
resumed_result = RollingFCRStudyResult.from_dict(resumed_bundle["profiles"][0]["result"])
roundtrip_result = RollingFCRStudyResult.from_dict(authored_result.to_dict())
resume_identity_table = pd.DataFrame([
    {"check": "canonical result round trip", "retained dates": len(roundtrip_result.records), "original fingerprint": authored_result.semantic_fingerprint, "restored fingerprint": roundtrip_result.semantic_fingerprint, "compatible": authored_result.semantic_fingerprint == roundtrip_result.semantic_fingerprint},
    {"check": "two-day checkpoint resumed to completion", "retained dates": len(resumed_result.records), "original fingerprint": authored_result.semantic_fingerprint, "restored fingerprint": resumed_result.semantic_fingerprint, "compatible": authored_result.semantic_fingerprint == resumed_result.semantic_fingerprint},
])
assert partial_result.completeness_status == "partial" and len(partial_result.records) == 2
assert resume_identity_table["compatible"].all()
display(failure_resume_table)
display(failure_transition_table)
display(resume_identity_table)

## 10. Optional native bounded reference benchmark

This branch is disabled by default because reference-input mixed-binary solves may run to their declared limits. It reads only the committed normalized fixture, uses HiGHS unless explicitly changed in the example command, and reports field-level outcomes such as exact displayed-decision parity, source-precision compatibility, source-incumbent equivalence, improved feasible incumbent compatible with the source bound, incompatibility, and unavailable evidence. Different time-limited incumbents are never called alternative optima merely because their objectives are close.

In [ ]:
native_reference_benchmark = {"status": "not_requested"}
if RUN_NATIVE_REFERENCE_BENCHMARK:
    from examples.fcr_rolling_backtest.run_reference_benchmark import run_bounded_reference_benchmark

    benchmark_payload = run_bounded_reference_benchmark(
        fixture_path=REPO_ROOT / REFERENCE_FIXTURE_PATH,
        profile_ids=(DEFAULT_PROFILE_ID,),
        dates=BOUNDED_DATES,
        solver_name="highs",
        time_limit_seconds=REFERENCE_BENCHMARK_TIME_LIMIT_SECONDS,
    )
    native_reference_benchmark = {"status": "completed", "payload": benchmark_payload}
display(pd.DataFrame([{"native reference benchmark": native_reference_benchmark["status"]}]))

## 11. Optional explicit-path external activation evidence

The complete ten-second regulation signal is not redistributed. The notebook never searches for or downloads it. When `RUN_EXTERNAL_ACTIVATION_REPLAY` is enabled, the next cell verifies only the explicitly configured file's governed byte count and SHA-256. The full four-day source/native replay remains the dedicated Feature 012-owned command shown in the result table; keeping the 529 MB parser out of the notebook avoids a second replay or settlement implementation. Missing data is explicit and does not weaken the offline optimization evidence.

In [ ]:
external_activation_result = {
    "status": "not_requested",
    "governed replay command": "uv run pytest tests/integration/simulator/test_fcr_rolling_external_reference_replay.py -m external_reference_data -q -s",
}
if RUN_EXTERNAL_ACTIVATION_REPLAY:
    activation_path = REPO_ROOT / EXTERNAL_ACTIVATION_CSV
    if not activation_path.is_file() or activation_path.is_symlink():
        raise FileNotFoundError(f"explicit governed activation CSV is unavailable: {activation_path}")
    expected_digest = "0c2bf5203410f193c555460ec5593b546af42556f29afac821584e403c407d13"
    expected_bytes = 528_805_941
    digest = hashlib.sha256()
    with activation_path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    if activation_path.stat().st_size != expected_bytes or digest.hexdigest() != expected_digest:
        raise ValueError("external activation file differs from the governed descriptor")
    external_activation_result = {
        "status": "verified_explicit_input",
        "path": str(EXTERNAL_ACTIVATION_CSV),
        "byte count": expected_bytes,
        "sha256": expected_digest,
        "governed replay command": external_activation_result["governed replay command"],
    }
display(pd.DataFrame([external_activation_result]))

In [ ]:
exported_figure_paths = []
if EXPORT_FIGURES:
    export_directory = REPO_ROOT / FIGURE_EXPORT_DIR
    export_directory.mkdir(parents=True, exist_ok=True)
    for figure_name, figure in sorted(generated_figures.items()):
        destination = export_directory / f"{figure_name}.png"
        figure.savefig(destination, dpi=150, bbox_inches="tight", metadata={"Software": "Open Energy Lab"})
        exported_figure_paths.append(destination)
display(pd.DataFrame({"exported figure": [str(path) for path in exported_figure_paths]}))

In [ ]:
notebook_display_payload = {
    "profile_rows": profile_table.to_dict(orient="records"),
    "authored_result_fingerprint": authored_result.semantic_fingerprint,
    "authored_daily_rows": authored_daily_table.to_dict(orient="records"),
    "carry_evidence": {key: value.evidence_fingerprint for key, value in carry_evidence.items()},
    "overlap_record_fingerprint": overlap_record.record_fingerprint,
    "causal_prefix_fingerprint": observed_a_fingerprint,
    "reference_fixture_fingerprint": reference_fixture.fixture_fingerprint,
    "figure_names": sorted(generated_figures),
}
notebook_display_fingerprint = stable_fingerprint(notebook_display_payload)
display(pd.DataFrame([{"notebook display fingerprint": notebook_display_fingerprint, "figures": len(generated_figures)}]))

## 12. Conclusions and remaining scope

The default walkthrough builds all five executable profiles, solves and evaluates a real four-day midnight-carry study, reconciles successor state, separates optimization from realized economics, visualizes the 08:00 accepted overlap, checks a half-open causal cutoff, retains lifecycle edge dates, and inspects all twenty bounded source records with explicit units and evidence strength. Every physical and settlement value in the authored study comes from Feature 012.

The normalized source evidence is a bounded oracle, not a proof of complete Julia trace or full-history parity. Displayed three-decimal values are intervals; time-limited runs are incumbents plus bounds; source feasibility needs an independent audit; optimizer-selected bids are assumed accepted rather than proven accepted; and unresolved workbook/paper mapping prevents a Table A1 claim. Intraday recourse, Experiments 6–9, degradation cost, TPM bid selection, live market acquisition, and mandatory full-history execution remain out of scope.